# joint_contrastive: contrastive loss trained together with the distillation

Instead of a contrastive stage after distillation (neutral: plain InfoNCE forgot, 67.11; with the anchor 67.74 vs 67.71), the contrastive loss runs **inside** the distillation: every `joint.every`-th step a batch of 128 (question, passage) pairs from the three QA datasets adds `joint.weight` x InfoNCE (scale 20, in-batch negatives) to the distillation gradient before the optimiser step, so both losses form one objective (`joint` config section, `src/turkish_distill/joint.py`). Idea: the geometry is still being built, so there is nothing to forget, and the pair labels act as an outside referee where the two teachers disagree.

Runs (all: 1k-base student, 200K mix, BGE-M3 + plain 8K-embed, lr 1e-4, 5400 steps, seed 42; pairs `pairs_train_clean.jsonl`, 38,259 pairs = ~298 batches per epoch):
1. `joint_w01_e4`: weight 0.1, every 4 steps (1350 pair batches, ~4.5 epochs over the pairs)
2. `joint_w03_e4`: weight 0.3, every 4 steps
3. `joint_w01_late`: weight 0.1, every 2 steps, only in the last 30% (from step 3780: 810 pair batches, ~2.7 epochs): the "contrastive at the end" idea while distillation still runs.

The reference is `scale_two_lr1e4_mix200k_s5400` (TR-MTEB 67.16; seed 43: 67.25), so only differences above 0.3 are read. Expect a profile shift (retrieval up, STS / clustering / pair classification possibly down); read the categories, not just the mean. Logs: `train/ctr_loss|ctr_nce|ctr_acc` next to the distillation terms, `val/mrr@10` as before (not a selection metric). A run takes ~10-15% longer than the plain one. An answer-term variant (`joint.answer_weight_mode=words` with `pairs_train_clean_ans.jsonl`, made by `contrastive_finetune.ipynb`) is commented out in `RUNS`. Re-running after a disconnect resumes the unfinished run and skips finished ones; the last cell releases the runtime. Delete this notebook when the results are in.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook, one after another (all: two-teacher config, 1k-base student, 200K mix, lr 1e-4, 5400 steps, seed 42).
# (name, config, extra overrides). A later override wins.
CONFIG = 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml'
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.max_steps=5400", "train.lr=0.0001"]
PAIRS = f'{DRIVE_ROOT}/data/stage1/pairs_train_clean.jsonl'
PAIRS_ANS = f'{DRIVE_ROOT}/data/stage1/pairs_train_clean_ans.jsonl'   # made by contrastive_finetune.ipynb (answers cell)
JOINT = [f'joint.pairs={PAIRS}', 'joint.batch_size=128']
RUNS = [
    ('joint_w01_e4', CONFIG, JOINT + ['joint.weight=0.1', 'joint.every=4']),
    ('joint_w03_e4', CONFIG, JOINT + ['joint.weight=0.3', 'joint.every=4']),
    ('joint_w01_late', CONFIG, JOINT + ['joint.weight=0.1', 'joint.every=2', 'joint.start_step=3780']),
    # answer term too (only if one of the above helps):
    # ('joint_w01_e4_ans', CONFIG, [f'joint.pairs={PAIRS_ANS}', 'joint.batch_size=128', 'joint.weight=0.1', 'joint.every=4',
    #                               'joint.answer_weight_mode=words', 'joint.answer_ref_words=20']),
]
assert os.path.exists(PAIRS), PAIRS

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        print(f"    '{name}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{os.path.basename(os.path.dirname(d))}/final'),")

In [ ]:
from google.colab import runtime
runtime.unassign()